# Week 3 Assignment
## SQL Analysis Using Subqueries, CTEs, and Window Functions

### Objective
Analyze Superstore sales data using advanced SQL concepts such as:
- Subqueries
- Common Table Expressions (CTEs)
- Window Functions
- Joins
- Aggregations


In [10]:
import pandas as pd
import sqlite3

# SECTION 1: Data Setup
    Query 1: View Imported Data

In [12]:
df = pd.read_csv("Superstore.csv", encoding='latin1')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [14]:
print(df.shape)
df.info()

(9994, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 1

In [16]:
conn = sqlite3.connect("superstore.db")
cursor = conn.cursor()

In [18]:
df.to_sql(
    "superstore_raw",
    conn,
    if_exists="replace",
    index=False
)

print("Table created successfully")

Table created successfully


In [20]:
cursor.execute("PRAGMA table_info(superstore_raw)")
print(cursor.fetchall())

[(0, 'Row ID', 'INTEGER', 0, None, 0), (1, 'Order ID', 'TEXT', 0, None, 0), (2, 'Order Date', 'TEXT', 0, None, 0), (3, 'Ship Date', 'TEXT', 0, None, 0), (4, 'Ship Mode', 'TEXT', 0, None, 0), (5, 'Customer ID', 'TEXT', 0, None, 0), (6, 'Customer Name', 'TEXT', 0, None, 0), (7, 'Segment', 'TEXT', 0, None, 0), (8, 'Country', 'TEXT', 0, None, 0), (9, 'City', 'TEXT', 0, None, 0), (10, 'State', 'TEXT', 0, None, 0), (11, 'Postal Code', 'INTEGER', 0, None, 0), (12, 'Region', 'TEXT', 0, None, 0), (13, 'Product ID', 'TEXT', 0, None, 0), (14, 'Category', 'TEXT', 0, None, 0), (15, 'Sub-Category', 'TEXT', 0, None, 0), (16, 'Product Name', 'TEXT', 0, None, 0), (17, 'Sales', 'REAL', 0, None, 0), (18, 'Quantity', 'INTEGER', 0, None, 0), (19, 'Discount', 'REAL', 0, None, 0), (20, 'Profit', 'REAL', 0, None, 0)]


In [22]:
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


# Query 2: Create Customers Table

In [28]:
cursor.execute("SELECT * FROM superstore_raw LIMIT 1")
print([desc[0] for desc in cursor.description])

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


# Query 3: Create Products Table

In [42]:
cursor.execute("""
CREATE TABLE products AS
SELECT DISTINCT
    "Product ID" AS product_id,
    Category AS category,
    "Sub-Category" AS sub_category,
    "Product Name" AS product_name
FROM superstore_raw
""")

conn.commit()

In [30]:
cursor.execute("PRAGMA table_info(superstore_raw)")
for col in cursor.fetchall():
    print(col)

(0, 'Row ID', 'INTEGER', 0, None, 0)
(1, 'Order ID', 'TEXT', 0, None, 0)
(2, 'Order Date', 'TEXT', 0, None, 0)
(3, 'Ship Date', 'TEXT', 0, None, 0)
(4, 'Ship Mode', 'TEXT', 0, None, 0)
(5, 'Customer ID', 'TEXT', 0, None, 0)
(6, 'Customer Name', 'TEXT', 0, None, 0)
(7, 'Segment', 'TEXT', 0, None, 0)
(8, 'Country', 'TEXT', 0, None, 0)
(9, 'City', 'TEXT', 0, None, 0)
(10, 'State', 'TEXT', 0, None, 0)
(11, 'Postal Code', 'INTEGER', 0, None, 0)
(12, 'Region', 'TEXT', 0, None, 0)
(13, 'Product ID', 'TEXT', 0, None, 0)
(14, 'Category', 'TEXT', 0, None, 0)
(15, 'Sub-Category', 'TEXT', 0, None, 0)
(16, 'Product Name', 'TEXT', 0, None, 0)
(17, 'Sales', 'REAL', 0, None, 0)
(18, 'Quantity', 'INTEGER', 0, None, 0)
(19, 'Discount', 'REAL', 0, None, 0)
(20, 'Profit', 'REAL', 0, None, 0)


# Query 4: Create Orders Table

In [48]:
cursor.execute("""
CREATE TABLE orders AS
SELECT
    "Order ID" as order_id,
    "Customer ID" as customer_id,
    "Product ID" as product_id,
    Sales as sales,
    Quantity as quantity,
    Profit as profit,
    "Order Date" as order_date
FROM superstore_raw
""")

conn.commit()

# SECTION 2: Required Queries
# Q1. Orders Above Average Sales

In [50]:
query = """
SELECT *
FROM orders
WHERE Sales >
(
    SELECT AVG(Sales)
    FROM orders
)
"""

pd.read_sql(query, conn).head()

,order_id,customer_id,product_id,sales,quantity,profit,order_date
0,CA-2016-152156,CG-12520,FUR-BO-10001798,261.9600,2,41.9136,11/8/2016
1,CA-2016-152156,CG-12520,FUR-CH-10000454,731.9400,3,219.5820,11/8/2016
2,US-2015-108966,SO-20335,FUR-TA-10000577,957.5775,5,-383.0310,10/11/2015
3,CA-2014-115812,BH-11710,TEC-PH-10002275,907.1520,6,90.7152,6/9/2014
4,CA-2014-115812,BH-11710,FUR-TA-10001539,1706.1840,9,85.3092,6/9/2014


# Q2. Highest Sales Order Per Customer

In [56]:
query = """
SELECT *
FROM orders o
WHERE Sales =
(
    SELECT MAX(Sales)
    FROM orders
    WHERE Customer_ID = o.Customer_ID
)
"""

pd.read_sql(query, conn)

,order_id,customer_id,product_id,sales,quantity,profit,order_date
0,CA-2016-152156,CG-12520,FUR-CH-10000454,731.9400,3,219.5820,11/8/2016
1,US-2015-108966,SO-20335,FUR-TA-10000577,957.5775,5,-383.0310,10/11/2015
2,CA-2014-115812,BH-11710,FUR-TA-10001539,1706.1840,9,85.3092,6/9/2014
3,CA-2015-106320,EB-13870,FUR-TA-10000577,1044.6300,3,240.2649,9/25/2015
4,US-2015-150630,TB-21520,FUR-BO-10004834,3083.4300,7,-1665.0522,9/17/2015
...,...,...,...,...,...,...,...
790,CA-2015-159534,DH-13075,OFF-BI-10003656,1087.9360,8,353.5792,3/20/2015
791,CA-2016-129630,IM-15055,TEC-CO-10003763,2799.9600,5,944.9865,9/4/2016
792,CA-2017-121559,HW-14935,OFF-AP-10002945,2405.2000,8,793.7160,6/1/2017
793,CA-2017-153871,RB-19435,OFF-BI-10004600,735.9800,2,331.1910,12/11/2017


# Q3. Total Sales Per Customer (CTE)

In [60]:
query = """
WITH customer_sales AS
(
    SELECT
        Customer_ID,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)

SELECT *
FROM customer_sales
ORDER BY Total_Sales DESC
"""

pd.read_sql(query, conn)

,Customer_ID,Total_Sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571
...,...,...
788,RS-19870,22.328
789,MG-18205,16.739
790,CJ-11875,16.520
791,LD-16855,5.304


# Q4. Customers Above Average Total Sales

In [32]:
query ="""
WITH customer_sales AS
(
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)

SELECT *
FROM customer_sales
WHERE total_sales >
(
    SELECT AVG(total_sales)
    FROM customer_sales
);
"""
pd.read_sql(query,conn)

,customer_id,total_sales
0,AA-10315,5563.560
1,AA-10645,5086.935
2,AB-10060,7755.620
3,AB-10105,14473.571
4,AC-10450,5527.846
...,...,...
289,VW-21775,6134.038
290,WB-21850,6160.102
291,YC-21895,5454.350
292,YS-21880,6720.444


# Q5. Rank Customers by Total Sales

In [35]:
query  = """
WITH customer_sales AS
(
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)

SELECT
    customer_id,
    total_sales,
    RANK() OVER
    (
        ORDER BY total_sales DESC
    ) AS sales_rank
FROM customer_sales;
"""
pd.read_sql(query,conn)

,customer_id,total_sales,sales_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


# Q6. ROW_NUMBER Per Customer

In [38]:
query = """
SELECT 
    customer_id,
    order_id,
    sales,
    ROW_NUMBER() over (
            partition by customer_id order by sales desc) 
            as row_num
    FROM orders
    """
pd.read_sql(query,conn)

,customer_id,order_id,sales,row_num
0,AA-10315,CA-2016-103982,3930.072,1
1,AA-10315,CA-2014-128055,673.568,2
2,AA-10315,CA-2016-103982,431.976,3
3,AA-10315,CA-2017-147039,362.940,4
4,AA-10315,CA-2014-128055,52.980,5
...,...,...,...,...
9989,ZD-21925,CA-2017-141481,61.440,5
9990,ZD-21925,CA-2014-143336,22.720,6
9991,ZD-21925,US-2016-147991,16.720,7
9992,ZD-21925,CA-2016-152471,15.984,8


In [68]:
query = """
SELECT
    Customer_ID,
    SUM(Sales) AS Total_Sales,
    RANK() OVER
    (
        ORDER BY SUM(Sales) DESC
    ) AS Sales_Rank
FROM orders
GROUP BY Customer_ID
"""

pd.read_sql(query, conn)

,customer_id,Total_Sales,Sales_Rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


# Q7. Top 3 Customers

In [46]:
query = """
WITH customer_sales AS
(
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)

SELECT 
    customer_id,
    total_sales
FROM customer_sales
ORDER BY total_sales DESC
LIMIT 3;
"""
pd.read_sql(query,conn)

,customer_id,total_sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339


# SECTION 3: Final Combined Query

In [41]:
query = """
WITH customer_sales AS
(
    SELECT
        Customer_ID,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)

SELECT
    c.Customer_Name,
    cs.Total_Sales,
    RANK() OVER
    (
        ORDER BY cs.Total_Sales DESC
    ) AS Customer_Rank

FROM customer_sales cs
JOIN customers c
ON cs.Customer_ID = c.Customer_ID
ORDER BY customer_rank;
"""

pd.read_sql(query, conn).head(20)

,customer_name,Total_Sales,Customer_Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
5,Ken Lonsdale,14175.229,6
6,Sanjit Chand,14142.334,7
7,Hunter Lopez,12873.298,8
8,Sanjit Engle,12209.438,9
9,Christopher Conant,12129.072,10


# SECTION 4: Mini Project Questions

# Top 5 Customers

In [50]:
query = """
WITH customer_sales AS
(
    SELECT
        Customer_ID,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)

SELECT *
FROM customer_sales
ORDER BY Total_Sales DESC
LIMIT 5
"""

pd.read_sql(query, conn)

,Customer_ID,Total_Sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571


# Bottom 5 Customers

In [53]:
query = """
WITH customer_sales AS
(
    SELECT
        Customer_ID,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)

SELECT *
FROM customer_sales
ORDER BY Total_Sales ASC
LIMIT 5
"""

pd.read_sql(query, conn)

,Customer_ID,Total_Sales
0,TS-21085,4.833
1,LD-16855,5.304
2,CJ-11875,16.520
3,MG-18205,16.739
4,RS-19870,22.328


# Customers With One Order

In [58]:
query = """
SELECT
    customer_id,
    COUNT(order_id) AS total_orders
FROM orders
GROUP BY customer_id
HAVING COUNT(order_id) = 1;
"""
pd.read_sql(query,conn)

,customer_id,total_orders
0,AO-10810,1
1,CJ-11875,1
2,JR-15700,1
3,LD-16855,1
4,RE-19405,1


# Highest Order Value Per Customer

In [61]:
query ="""
SELECT
    customer_id,
    MAX(sales) AS highest_order_value
FROM orders
GROUP BY customer_id;
"""
pd.read_sql(query,conn)

,customer_id,highest_order_value
0,AA-10315,3930.072
1,AA-10375,499.980
2,AA-10480,479.970
3,AA-10645,1323.900
4,AB-10015,341.960
...,...,...
788,XP-21865,337.088
789,YC-21895,2934.330
790,YS-21880,2793.528
791,ZC-21910,1516.200


# Week 3 Insights

1. A small number of customers generate most of the sales.
2. Several customers have only one order.
3. Top customers contribute significantly more revenue than average customers.
4. Window functions simplify ranking and order analysis.
5. CTEs make complex aggregation queries easier to read.
6. Customers above average sales can be targeted for premium offerings.